In [2]:
from fst_runtime.fst import Fst
from cg3_process import disambiguate
FST                    = Fst("../data/fst/ojibwe.att")  
CG3_GRAMMAR_PATH       = "../data/CG3_rules/Ojibwe_disambiguation.cg3"

In [14]:
"""
cg3_stats.py  -  diagnostic wrapper for Ojibwe FST + CG3
"""

from __future__ import annotations
from collections import Counter
from typing import Dict, List, Tuple

from fst_runtime.fst import Fst
from cg3_process import (
    PUNCTUTATIONS,
    PRESERVE_TOKEN,
    ojibwe_sentence_to_cg3_format,
    cg3_process_text,
)
from tabulate import tabulate

# ------------------------------------------------------------------ #
#   Tag inventories
# ------------------------------------------------------------------ #
VERB_TAGS    = {"VTA", "VAI", "VTI", "VII", "VAIO"}
PRONOUN_TAGS = {"PRONDem", "PRONDub", "PRONIndf", "PRONInter",
                "PRONPret", "PRONSim", "PRONPer"}
NOUN_TAGS    = {"NA", "NI", "NAD", "NID"}
ADVERB_TAGS  = {"ADVConj", "ADVDisc", "ADVDub", "ADVGram", "ADVInter",
                "ADVLoc", "ADVMan", "ADVNeg", "ADVPred", "ADVQnt",
                "ADVTmp", "AVDDeg"}
WORD_TYPES   = ("verb", "pronoun", "noun", "adverb", "other")

# Tags never considered part of the morphological signature
_ALWAYS_DISCARD = VERB_TAGS | NOUN_TAGS | ADVERB_TAGS | PRONOUN_TAGS


# ------------------------------------------------------------------ #
#   Tiny helpers
# ------------------------------------------------------------------ #
def _major_pos(tags: set[str]) -> str:
    if tags & PRONOUN_TAGS: return "pronoun"
    if tags & VERB_TAGS:    return "verb"
    if tags & NOUN_TAGS:    return "noun"
    if tags & ADVERB_TAGS:  return "adverb"
    return "other"


def _strip_common(tagsets: List[set[str]]) -> List[set[str]]:
    """Remove tags shared by *all* sets."""
    return tagsets if not tagsets else [ts - set.intersection(*tagsets)
                                        for ts in tagsets]


def _count_readings(block: str) -> Tuple[int, int]:
    """Return (#words, #analyses) ignoring punctuation."""
    words = analyses = 0
    for ln in block.splitlines():
        if ln.startswith('"<') and ln.endswith('>"'):
            tok = ln[2:-2]
            if tok and tok not in PUNCTUTATIONS and tok != PRESERVE_TOKEN:
                words += 1
        elif ln.startswith("\t"):
            analyses += 1
    return words, analyses


def _count_by_type(block: str) -> Dict[str, Dict[str, int]]:
    counts = {wt: {"words": 0, "readings": 0} for wt in WORD_TYPES}
    cur_pos: set[str] = set()
    in_word = False

    for ln in block.splitlines():
        if ln.startswith('"<') and ln.endswith('>"'):
            for wt in cur_pos:
                counts[wt]["words"] += 1
            cur_pos.clear()

            tok = ln[2:-2]
            in_word = bool(tok and tok not in PUNCTUTATIONS
                                and tok != PRESERVE_TOKEN)

        elif in_word and ln.startswith("\t"):
            tags = set(ln.strip().split()[1:])
            wt = _major_pos(tags)
            cur_pos.add(wt)
            counts[wt]["readings"] += 1

    for wt in cur_pos:
        counts[wt]["words"] += 1
    return counts


# ------------------------------------------------------------------ #
#   Ambiguity analysis
# ------------------------------------------------------------------ #
def _classify_amb(readings: List[List[str]]) -> str:
    """Return 'lemma' | 'preverb' | 'pos' | 'morpho'."""
    lemmas = {r[0].strip('"') for r in readings if r[0].startswith('"')}
    if len(lemmas) > 1:
        return "lemma"

    pv_sets = [{t for t in r[1:] if t.startswith("PV")} for r in readings]
    if len({frozenset(s) for s in pv_sets}) > 1:
        return "preverb"

    pos_set = {_major_pos(set(r[1:])) for r in readings}
    if len(pos_set) > 1:
        return "pos"

    return "morpho"


def _ambiguity(block: str, total_words: int,
               top_n: int = 50) -> Dict[str, object]:
    kinds, tok_counter = Counter(), Counter()
    patterns = {"lemma": {}, "preverb": {}, "morpho": {}, "pos": {}, }
    token, readings = None, []

    for ln in block.splitlines() + ["## END"]:
        if ln.startswith('"<') and ln.endswith('>"'):
            if token and len(readings) > 1:
                k = _classify_amb(readings)
                pos_bucket = _major_pos(set(readings[0][1:]))
                kinds[k] += 1
                tok_counter[token] += 1

                # pattern key
                if k == "pos":
                    key = "+".join(sorted({_major_pos(set(r[1:])) for r in readings}))

                elif k == "preverb":
                    pv_sets = [{t for t in r[1:] if t.startswith("PV")}
                               for r in readings]
                    diff = _strip_common(pv_sets)
                    key = " vs ".join(sorted(",".join(sorted(d)) if d else "{}"
                                             for d in diff))

                elif k == "lemma":
                    key = " vs ".join(sorted({r[0].strip('"') for r in readings}))

                else:  # morpho
                    sig_sets = [ {t for t in r[1:] if not t.startswith("PV")
                                                  and t not in _ALWAYS_DISCARD}
                                 for r in readings ]
                    diff = _strip_common(sig_sets)
                    key = " | ".join(sorted(",".join(sorted(d)) if d else "{}"
                                             for d in diff))

                if k == "pos":
                    store = patterns["pos"]
                else:
                    store = patterns[k].setdefault(pos_bucket, {})
                info = store.setdefault(key, {"count": 0, "tokens": Counter()})
                info["count"] += 1
                info["tokens"][token] += 1

            token, readings = ln[2:-2], []
        elif ln.startswith("\t"):
            readings.append(ln.strip().split())

    total = sum(kinds.values())
    overview = {"total_ambiguous_tokens": total,
                "pct_tokens_ambiguous": total / total_words if total_words else 0.0,
                **{t: kinds[t] for t in ("lemma", "preverb", "pos", "morpho")}}

    return {"overview": overview,
            "top_tokens": tok_counter.most_common(top_n),
            "patterns": patterns}


# ------------------------------------------------------------------ #
#   Table helpers
# ------------------------------------------------------------------ #
def _table_dict(stats: dict) -> Dict[str, str]:
    summary = tabulate(
        [["total words", stats["total_words"]],
         ["readings before", stats["total_readings_before"]],
         ["readings after", stats["total_readings_after"]],
         ["removed", stats["analyses_removed"]],
         ["pct removed", f'{stats["pct_removed"]:.3f}'],
         ["ambiguous tokens", stats["ambiguity"]["overview"]["total_ambiguous_tokens"]],
         ["pct ambiguous", f'{stats["ambiguity"]["overview"]["pct_tokens_ambiguous"]:.3f}']],
        tablefmt="github")

    by_type = tabulate(
        [[wt,
          d["words"], d["readings_before"], d["readings_after"],
          d["analyses_removed"], f'{d["avg_before"]:.2f}', f'{d["avg_after"]:.2f}']
         for wt, d in stats["by_type"].items()],
        headers=["type", "words", "before", "after", "removed", "avg_b", "avg_a"],
        tablefmt="github")

    ov = stats["ambiguity"]["overview"]
    amb_ov = tabulate(
        [["lemma amb.",       ov["lemma"]],
         ["preverb amb.",     ov["preverb"]],
         ["POS amb.",         ov["pos"]],
         ["morpho amb.",      ov["morpho"]]],
        tablefmt="github")

    amb_top = tabulate(stats["ambiguity"]["top_tokens"],
                       headers=["top 50 tokens", "count"], tablefmt="github")

    return {"summary": summary, "by_type": by_type,
            "amb_ov": amb_ov, "amb_top": amb_top}


def _pattern_table(pat_dict: Dict[str, dict], top: int = 200) -> str:
    """If pat_dict is nested {pos → {pattern → info}}, expand it first."""
    if pat_dict and isinstance(next(iter(pat_dict.values())), dict) \
       and "count" not in next(iter(pat_dict.values())).keys():
        rows = []
        for pos, sub in pat_dict.items():
            for k, info in sub.items():
                rows.append([pos, k, info["count"],
                             ", ".join(t for t,_ in info["tokens"].most_common(3))])
        hdr = ["POS", "pattern", "tokens", "examples ≤3"]
    else:   # flat dict as before
        rows = [[k, info["count"],
                 ", ".join(t for t,_ in info["tokens"].most_common(3))]
                for k, info in pat_dict.items()]
        hdr = ["pattern", "tokens", "examples ≤3"]

    rows = sorted(rows, key=lambda r: r[2 if hdr[0]=="POS" else 1], reverse=True)[:top]
    return tabulate(rows, headers=hdr, tablefmt="github") if rows else ""


# ------------------------------------------------------------------ #
#   Public API
# ------------------------------------------------------------------ #
def disambiguate_with_stats(text: str,
                            grammar: str,
                            fst: Fst,
                            *,
                            verbose: bool = False
                           ) -> Tuple[str, Dict]:
    """Return (CG3-after block, statistics dict)."""
    before = ojibwe_sentence_to_cg3_format(text, fst=fst)
    w_b, a_b = _count_readings(before)
    type_b   = _count_by_type(before)

    after = cg3_process_text(before, grammar)
    w_a, a_a = _count_readings(after)
    type_a   = _count_by_type(after)

    removed = a_b - a_a
    stats = {
        "total_words": w_a,
        "total_readings_before": a_b,
        "total_readings_after": a_a,
        "analyses_removed": removed,
        "pct_removed": removed / a_b if a_b else 0.0,
        "by_type": {},
    }

    for wt in WORD_TYPES:
        wb, rb = type_b[wt]["words"], type_b[wt]["readings"]
        ra     = type_a[wt]["readings"]
        stats["by_type"][wt] = {
            "words": wb,
            "readings_before": rb,
            "readings_after":  ra,
            "analyses_removed": rb - ra,
            "avg_before": rb / wb if wb else 0.0,
            "avg_after":  ra / wb if wb else 0.0,
        }

    stats["ambiguity"] = _ambiguity(after, w_a)

    if verbose:
        tbl = _table_dict(stats)
        pat = stats["ambiguity"]["patterns"]
        sections = [
            tbl["summary"], tbl["by_type"], tbl["amb_ov"], tbl["amb_top"]
        ]
        sections += [
            "# POS-level patterns",      _pattern_table(pat["pos"]),
            "# Lemma patterns",          _pattern_table(pat["lemma"]),
            "# Preverb patterns",        _pattern_table(pat["preverb"]),
            "# Morphological patterns",  _pattern_table(pat["morpho"])
        ]
        print("\n\n".join(s for s in sections if s))

    return after, stats

In [17]:
with open("../data/parallel_data/treebank_sentences/ojibwe_all.txt", encoding="utf-8") as fh:
    corpus = fh.read()


In [ ]:

_, stats = disambiguate_with_stats(
    text=corpus,
    grammar=CG3_GRAMMAR_PATH,
    fst=FST,
    verbose=True, 
)

|------------------|---------|
| total words      | 208     |
| readings before  | 309     |
| readings after   | 248     |
| removed          |  61     |
| pct removed      |   0.197 |
| ambiguous tokens |  43     |
| pct ambiguous    |   0.207 |

| type    |   words |   before |   after |   removed |   avg_b |   avg_a |
|---------|---------|----------|---------|-----------|---------|---------|
| verb    |      81 |      152 |     111 |        41 |    1.88 |    1.37 |
| pronoun |      27 |       41 |      33 |         8 |    1.52 |    1.22 |
| noun    |      50 |       71 |      60 |        11 |    1.42 |    1.2  |
| adverb  |      33 |       34 |      33 |         1 |    1.03 |    1    |
| other   |      11 |       11 |      11 |         0 |    1    |    1    |

|--------------|----|
| lemma amb.   |  4 |
| preverb amb. | 12 |
| POS amb.     |  0 |
| morpho amb.  | 27 |

| top 50 tokens             |   count |
|---------------------------|---------|
| iniwen                    |     

In [18]:
from pathlib import Path
from io import StringIO
from contextlib import redirect_stdout

out_path = Path("../data/disambig_stats/ojibwe_stats.txt")
out_path.parent.mkdir(parents=True, exist_ok=True) 

buf = StringIO()
with redirect_stdout(buf):
    disambiguate_with_stats(
        text=corpus,
        grammar=CG3_GRAMMAR_PATH,
        fst=FST,
        verbose=True, 
    )

out_path.write_text(buf.getvalue(), encoding="utf-8")
print(f"Report written to {out_path.resolve()}")

Report written to /workspaces/Ojibwe_Constraint_Grammar/data/disambig_stats/ojibwe_stats.txt
